In [1]:
import pandas as pd
import torch
import anndata as ad
from scipy.spatial import Delaunay
import sys
spagfm_dir='/fs/ess/PAS1475/yzhong/sf_project/codebase/G2PM_spatial'
sys.path.append(spagfm_dir)
from stRoamer.utils.graph_build import filter_edge, edge_index_from_delaunay, pyg_obj
import scgpt.tasks as sct

/fs/ess/PAS1475/yzhong/sf_project/conda_env/new_scgpt/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-04 19:35:11,659 - datasets - INFO - PyTorch version 2.0.1 available.
2026-05-04 19:35:11,679 - datasets - INFO - JAX version 0.6.2 available.


In [2]:
file_path='/fs/ess/PAS1475/yzhong/sf_project/datasets/segmentation_noise_new'

In [3]:
sample_name_l=["Lung5_Rep1","Lung5_Rep2","Lung5_Rep3","Lung6",
               "Lung9_Rep1","Lung9_Rep2","Lung12","Lung13"]

for sample_name in sample_name_l:

    adata_dict = torch.load(file_path+f'/CosMx_Human_{sample_name}_noise_processed.pt')
    print(adata_dict)

    model_dir='/fs/ess/PAS1475/yzhong/sf_project/backup/ccv/datasets/scGPT_human'
    gene_col='index'
    adata_scgpt_graph_l = []
    threshold = 0.99

    for key, adata in adata_dict.items():
        # Node features
        adata = sct.embed_data(
                            adata,
                            model_dir,
                            gene_col=gene_col,
                            batch_size=512,
                            return_new_adata=False
                            )
        X_scgpt = adata.obsm["X_scGPT"]
        adata_scgpt_f = torch.tensor(X_scgpt, dtype=torch.float)

        # Build edges from Delaunay + filter
        tri = Delaunay(adata.obs[['new_x', 'new_y']])
        adata_edge = filter_edge(tri, threshold=threshold).simplices
        edge_index = edge_index_from_delaunay(adata_edge)  # must output [2, E]

        # Build pyg graph
        adata_scgpt_graph = pyg_obj(adata_scgpt_f, edge_index)

        # # Add labels
        adata_scgpt_graph.niche = torch.tensor(
            pd.Categorical(adata.obs['niche']).codes,
            dtype=torch.long
        )
        adata_scgpt_graph.cell_type = torch.tensor(
            pd.Categorical(adata.obs['cell_type']).codes,
            dtype=torch.long
        )
        adata_scgpt_graph.dataset = key

        print(f'{key}: pyG graph', adata_scgpt_graph)
        adata_scgpt_graph_l.append(adata_scgpt_graph)

    torch.save(adata_scgpt_graph_l, file_path+f'/CosMx_Human_{sample_name}_noise_processed_og_graph_scgpt_test.pt')

{'Lung5_Rep1_0': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'raw_sdimx', 'raw_sdimy', 'raw_cell_ID', 'viz_sdimx', 'viz_sdimy', 'viz_cell_ID', 'cell_ID', 'fov', 'Area', 'AspectRatio', 'Width', 'Height', 'Mean.CD298', 'Max.CD298', 'Mean.G', 'Max.G', 'Mean.Y', 'Max.Y', 'Mean.R', 'Max.R', 'Mean.DAPI', 'Max.DAPI', 'dualfiles', 'Slide_name', 'tissue', 'Run_name', 'ISH.concentration', 'Dash', 'slide_ID_numeric', 'Run_Tissue_name', 'Panel', 'Diversity', 'totalcounts', 'log10totalcounts', 'background', 'remove_flagged_cells', 'patient', 'cell_type', 'niche', 'prop_tumor_in_100_neighbors', 'n_counts', 'leiden', 'new_x', 'new_y'
    var: 'feat_ID', 'remove_flagged_genes', 'n_cells'
    uns: 'log1p', 'pca', 'neighbors', 'umap', 'leiden'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'distances', 'connectivities', 'Lung5_Rep1_0.5': AnnData object with n_obs × n_vars = 90650 × 960
    obs: 'orig.ident', 'nCount_RN

Embedding cells: 100%|██████████| 178/178 [00:26<00:00,  6.71it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/new_scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung5_Rep1_0: pyG graph Data(x=[90650, 512], edge_index=[2, 533776], niche=[90650], cell_type=[90650], dataset='Lung5_Rep1_0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 178/178 [00:23<00:00,  7.61it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/new_scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung5_Rep1_0.5: pyG graph Data(x=[90650, 512], edge_index=[2, 532888], niche=[90650], cell_type=[90650], dataset='Lung5_Rep1_0.5')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


...

Lung13_1.0: pyG graph Data(x=[76421, 512], edge_index=[2, 450696], niche=[76421], cell_type=[76421], dataset='Lung13_1.0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 150/150 [00:21<00:00,  7.06it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/new_scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung13_2.0: pyG graph Data(x=[76421, 512], edge_index=[2, 452314], niche=[76421], cell_type=[76421], dataset='Lung13_2.0')
scGPT - INFO - match 958/960 genes in vocabulary of size 60697.


Embedding cells: 100%|██████████| 150/150 [00:21<00:00,  7.00it/s]
/fs/ess/PAS1475/yzhong/sf_project/conda_env/new_scgpt/lib/python3.10/site-packages/scgpt/tasks/cell_emb.py:279: ImplicitModificationWarning: Setting element `.obsm['X_scGPT']` of view, initializing view as actual.
  adata.obsm["X_scGPT"] = cell_embeddings


Lung13_5.0: pyG graph Data(x=[76421, 512], edge_index=[2, 454164], niche=[76421], cell_type=[76421], dataset='Lung13_5.0')
